# PyTorch Basics

<a target="_blank" href="https://colab.research.google.com/github/imamitjain/notebooks/blob/main/03-deep-learning/01_pytorch_basics.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective:** Learn PyTorch tensors, autograd, and the fundamental building blocks for neural networks — the framework used throughout the deep learning and LLM sections.

**Prerequisites:** NumPy essentials (Section 01)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q numpy matplotlib torch


In [ ]:
import torch
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Tensors — NumPy on Steroids

In [ ]:
# Creating tensors
a = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32)
print(f"Tensor: {a}, shape: {a.shape}, dtype: {a.dtype}")

# Common constructors
zeros = torch.zeros(3, 4)
ones = torch.ones(2, 3)
rand = torch.randn(3, 3)
arange = torch.arange(0, 10, 2)

print(f"Random 3x3:\n{rand}")
print(f"Arange: {arange}")

In [ ]:
# NumPy <-> Tensor conversion (shared memory!)
np_arr = np.array([1.0, 2.0, 3.0])
tensor_from_np = torch.from_numpy(np_arr)
back_to_np = tensor_from_np.numpy()

print(f"NumPy: {np_arr}")
print(f"Tensor: {tensor_from_np}")

# Modify numpy — tensor changes too
np_arr[0] = 999
print(f"After modifying numpy: tensor = {tensor_from_np}")

## 2. Operations and Broadcasting

In [ ]:
x = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
y = torch.tensor([[5, 6], [7, 8]], dtype=torch.float32)

print(f"x + y:\n{x + y}\n")
print(f"x * y (element-wise):\n{x * y}\n")
print(f"x @ y (matrix multiply):\n{x @ y}\n")

# Broadcasting
row = torch.tensor([10, 20])
print(f"x + row:\n{x + row}")

## 3. Autograd — Automatic Differentiation

This is what makes PyTorch a deep learning framework, not just a tensor library.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

# f(x) = x² + 2x + 1
y = x**2 + 2*x + 1

# Compute gradient: dy/dx = 2x + 2
y.backward()
print(f"x = {x.item()}")
print(f"f(x) = {y.item()}")
print(f"df/dx = {x.grad.item()}")
print(f"Expected: 2*{x.item()} + 2 = {2*x.item() + 2}")

In [ ]:
# Autograd with vectors
W = torch.randn(3, 2, requires_grad=True)
x = torch.randn(2)
b = torch.randn(3, requires_grad=True)

# Simple linear layer: y = Wx + b
y = W @ x + b
loss = y.sum()
loss.backward()

print(f"W.grad shape: {W.grad.shape}")
print(f"b.grad: {b.grad}")

## 4. Building a Simple Model (nn.Module)

In [ ]:
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)

model = SimpleNet(10, 32, 1)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training Loop Anatomy

In [ ]:
# Synthetic data
torch.manual_seed(42)
X = torch.randn(200, 10)
y = (X[:, 0] * 3 + X[:, 1] * -2 + torch.randn(200) * 0.5).unsqueeze(1)

# Model, loss, optimizer
model = SimpleNet(10, 32, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

losses = []
for epoch in range(100):
    y_pred = model(X)
    loss = criterion(y_pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.show()

## Try It Yourself

1. Modify `SimpleNet` to add dropout and a second hidden layer. Compare training loss with and without dropout.
2. Repeat the training loop but use `torch.optim.SGD` instead of Adam. Try different learning rates `[0.001, 0.01, 0.1]` and plot the loss curves on the same axes.
3. Move the model and data to the MPS device (if available) and verify training still works. Time the difference vs CPU for a larger dataset (10,000 samples).

In [ ]:
# Your code here